# CIFAR-10 com MLP — Relatório de experimentos (grid search em blocos)

**Metodologia.** Em vez de escolher um valor por vez, cada bloco executa um **grid search** e o vencedor é
decidido pelos dados. O campeão de um bloco é lido automaticamente pelo bloco seguinte, sem nenhum valor escolhido à mão:

| Bloco | Pergunta | Grid |
|---|---|---|
| 0 — Referência | Onde partimos? | regressão logística e MLP do notebook original |
| 1 — Topologia | Qual estrutura base absorve melhor o CIFAR-10? | camadas × neurônios |
| 2 — Otimização | Qual algoritmo e taxa de aprendizagem fazem a rede campeã convergir melhor? | otimizador × lr |
| Checagem | A escolha em blocos se sustenta? | 2º e 3º do Bloco 1 com o otimizador campeão |
| 3 — Regularização e erro | Como dropout e função de erro afetam a generalização? | dropout × função de erro |

**Protocolo de avaliação.**
- **Validação cruzada estratificada em 5 folds** sobre as 50.000 imagens de treino, com as mesmas partições em todos os experimentos (comparações pareadas).
- **Triagem:** cada configuração roda os folds 1–3; as 3 melhores **completam os folds 4–5** (sem refazer nada) e o campeão é a maior média nos 5 folds.
- **Escolha sempre pela validação.** O conjunto de teste (10.000 imagens) só é revelado na seção final, para os campeões. Usá-lo para escolher tornaria o número final otimista (vazamento de dados).
- Diferenças menores que o desvio padrão entre folds não devem ser tratadas como melhora real.

**Registro dos resultados.** Cada experimento grava em `/kaggle/working/outputs/{exp_name}/`:

| Arquivo | Conteúdo |
|---|---|
| `parametros.json` | hiperparâmetros exatos, arquitetura resolvida, versões e commit |
| `historico_treino.csv` | uma linha por **fold × época**: loss, acurácia, precision/recall/F1 (gerais e por classe) de treino e validação, gap |
| `historico_treino_agregado.csv` | média e desvio padrão entre folds, por época |
| `resultados.json` | resultado de cada fold e médias (validação na melhor época e teste) |
| `melhor_modelo.pth` | pesos com a menor `val/loss` (melhor fold); cada fold em `folds/fold_k/` |

Cada bloco grava em `outputs/_grids/{bloco}/`: `configs.csv`, `ranking_triagem.csv`, `ranking_final.csv`, `campeao.json` e `logs/`.
Tabelas e figuras dos relatórios ficam em `outputs/_relatorio/`. Tudo é gravado a cada época, antes do envio ao Weights & Biases.

**Tempo estimado:** ~4 h com 1 GPU T4 ou **~2–2,5 h com 2× T4** (estimativa a partir de medições; varia ±30%).
Os experimentos são distribuídos entre as GPUs visíveis (2 por GPU, ajustável em `WORKERS_PER_GPU`).

**Antes de executar (Kaggle):**
1. *Settings → Accelerator*: **GPU T4 ×2**. *Settings → Internet*: **On**.
2. *Add-ons → Secrets*: `GITHUB_TOKEN` (obrigatório se o repositório for privado) e `WANDB_API_KEY`
   (opcional — sem ele tudo continua salvo localmente). Marque os dois como anexados a este notebook.
3. Use **Save Version → Save & Run All (Commit)**: `/kaggle/working` (com `outputs/` e `outputs.zip`) fica salvo na aba *Output*.
4. **Retomada:** tudo é retomável. Se a sessão cair, adicione o Output da versão anterior como *Input*, preencha `RESUME_FROM`
   e rode de novo: experimentos e folds concluídos são pulados.

## 0. Preparação do ambiente

In [ ]:
import os

REPO_URL = "https://github.com/diegoflyra/dfal-neural-networks.git"
REPO_DIR = "/tmp/dfal-neural-networks"  # fora de /kaggle/working: código e dataset não poluem o Output

# Repositório privado: crie o secret GITHUB_TOKEN (Add-ons → Secrets) com um token de leitura do GitHub.
# O token fica só em /tmp (não vai para o Output) e nunca é impresso.
clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    clone_url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    print("GitHub: usando GITHUB_TOKEN")
except Exception:
    print("GitHub: sem GITHUB_TOKEN (funciona apenas se o repositório for público)")

if os.path.isdir(REPO_DIR):
    !git -C $REPO_DIR pull -q
else:
    !git clone -q $clone_url $REPO_DIR
%cd $REPO_DIR
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import shutil
import sys

import pandas as pd

WORKERS_PER_GPU = 2  # experimentos simultâneos por GPU
RESUME_FROM = ""  # ex.: "/kaggle/input/<output-da-versao-anterior>/outputs" para retomar

# Onde os resultados são gravados (lido por run_experiment.py, grid_search.py e report_utils.py)
os.environ["EXP_OUTPUT_DIR"] = "/kaggle/working/outputs"
os.makedirs(os.environ["EXP_OUTPUT_DIR"], exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, os.environ["EXP_OUTPUT_DIR"], dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

# Weights & Biases via Kaggle Secrets; se falhar, os experimentos seguem apenas com o registro local.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: chave carregada.")
except Exception as exc:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado ({exc}). Resultados continuam em {os.environ['EXP_OUTPUT_DIR']}.")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import report_utils as rep

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino):
# GPUs, flags → arquitetura, carregamento em GPU/K-fold (baixa o CIFAR-10) e construção de todas as configurações dos grids
!nvidia-smi -L
!python tests/check_hyperparams.py
!python tests/check_data_loader.py
!python tests/check_grids.py

## Bloco 0 — Referências

Mesmo protocolo dos blocos seguintes (Adam, lr=1e-3, batch 128, early stopping com paciência 5, 5 folds), aplicado à
regressão logística (piso) e à MLP 64-128-64 do notebook original. Servem de ponto de comparação para todo o estudo.

Fixos no bloco: `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=40`, `patience=5`, `eval_train=True`, `activation=relu`, `loss_fn=cross_entropy`, `dropout=0.0`. Todas as configurações rodam os 5 folds.

In [ ]:
!python src/grid_search.py grids/mlp_b0_referencia.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.grid_ranking("mlp_b0_referencia", "final")

## Bloco 1 — Topologia

**Pergunta:** qual combinação de profundidade (camadas ocultas) e largura (neurônios por camada) absorve melhor o CIFAR-10?

**Por que primeiro:** a topologia define a capacidade do modelo; otimização e regularização são ajustadas sobre ela.
Todas as redes usam a mesma otimização padrão e **nenhuma regularização**, com early stopping na `val/loss`
para que redes grandes não sejam penalizadas por treinar além do ponto ótimo.

| Eixo | Valores |
|---|---|
| `mlp_layers` | `1`, `2`, `3`, `4`, `5` |
| `mlp_neurons` | `128`, `256`, `512`, `1024`, `2048` |

**25 combinações.** Fixos no bloco: `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=40`, `patience=5`, `eval_train=True`, `activation=relu`, `loss_fn=cross_entropy`, `dropout=0.0`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

**O que observar:** o heatmap camadas × neurônios, o `gap/accuracy` (overfitting) das redes maiores e se ganhos de
capacidade continuam aparecendo na validação.

In [ ]:
!python src/grid_search.py grids/mlp_b1_topologia.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `mlp_b1_topologia`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("mlp_b1_topologia", "triagem")

In [ ]:
rep.heatmap("mlp_b1_topologia", row="mlp_layers", col="mlp_neurons")

In [ ]:
rep.plot_grid_bars("mlp_b1_topologia")

#### Confirmação das finalistas e campeão — `mlp_b1_topologia`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("mlp_b1_topologia", "final")

In [ ]:
rep.show_champion("mlp_b1_topologia")
rep.plot_finalists("mlp_b1_topologia")

In [ ]:
rep.heatmap("mlp_b1_topologia", row="mlp_layers", col="mlp_neurons", value="gap/accuracy_mean")

### 📝 Análise — Bloco 1

- **Profundidade ou largura: o que mais contribuiu?** _…_
- **A partir de qual tamanho a validação estagna?** _…_
- **Como o gap cresce com a capacidade?** _…_
- **Diferença do campeão para as referências (validação):** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 2 — Otimização

**Pergunta:** com a topologia campeã do Bloco 1 fixa, qual combinação de algoritmo e taxa de aprendizagem converge melhor?

**Base:** o campeão do Bloco 1 (abaixo), lido de `outputs/_grids/mlp_b1_topologia/campeao.json`. SGD usa momentum 0,9.
O grid de lr é comum aos dois algoritmos de propósito: ele mostra a faixa em que cada um funciona (lr baixos
não convergem no SGD; lr altos podem divergir no Adam — divergências são registradas sem perda de arquivos).

| Eixo | Valores |
|---|---|
| `optimizer` | `sgd`, `adam` |
| `lr` | `0.0001`, `0.0003`, `0.001`, `0.003`, `0.01`, `0.03`, `0.1` |

**14 combinações.** Herdado do campeão de: `mlp_b1_topologia`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

**O que observar:** o heatmap otimizador × lr, a `melhor_epoca_media` (velocidade) e as curvas das finalistas.

In [ ]:
rep.show_champion("mlp_b1_topologia")

In [ ]:
!python src/grid_search.py grids/mlp_b2_otimizacao.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `mlp_b2_otimizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("mlp_b2_otimizacao", "triagem")

In [ ]:
rep.heatmap("mlp_b2_otimizacao", row="optimizer", col="lr")

In [ ]:
rep.plot_grid_bars("mlp_b2_otimizacao")

#### Confirmação das finalistas e campeão — `mlp_b2_otimizacao`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("mlp_b2_otimizacao", "final")

In [ ]:
rep.show_champion("mlp_b2_otimizacao")
rep.plot_finalists("mlp_b2_otimizacao")

In [ ]:
rep.heatmap("mlp_b2_otimizacao", row="optimizer", col="lr", value="melhor_epoca_media")

### 📝 Análise — Bloco 2

- **Faixa de lr útil para SGD e para Adam:** _…_
- **Qual convergiu em menos épocas?** _…_
- **Houve divergência? Em quais combinações?** _…_
- **Ganho sobre o Bloco 1 (validação):** _…_

### Checagem de interação

A busca em blocos assume que a melhor topologia com Adam lr=1e-3 continua sendo a melhor com o otimizador campeão.
Para verificar, o 2º e o 3º colocados do Bloco 1 são treinados com o otimizador/lr campeões do Bloco 2 (5 folds).
O Bloco 3 parte da melhor rede entre o campeão do Bloco 2 e esta checagem.

Herdado do campeão de: `mlp_b2_otimizacao`. Todas as configurações rodam os 5 folds.

In [ ]:
!python src/grid_search.py grids/mlp_b2_checagem.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
pd.concat([rep.grid_ranking("mlp_b2_otimizacao", "final").head(1),
           rep.grid_ranking("mlp_b2_checagem", "final")], ignore_index=True)

### 📝 Análise — Checagem

- **A ordem das topologias se manteve com o novo otimizador?** _…_
- **Se mudou, o que isso indica sobre a interação topologia × otimização?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 3 — Regularização e função de erro

**Pergunta:** com a rede convergindo bem, como a taxa de dropout e a função de erro afetam a generalização?

**Base:** a melhor rede entre o campeão do Bloco 2 e a checagem. Épocas e paciência aumentadas, pois dropout
retarda a convergência. A seleção é pela `val/accuracy`: a `val/loss` de MSE e de entropia cruzada estão em
escalas diferentes e não são comparáveis entre si.

| Eixo | Valores |
|---|---|
| `dropout` | `0.0`, `0.2`, `0.3`, `0.5` |
| `loss_fn` | `cross_entropy`, `mse` |

**8 combinações.** Herdado do campeão de: `mlp_b2_otimizacao`, `mlp_b2_checagem`. Fixos no bloco: `epochs=50`, `patience=7`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

**O que observar:** o `gap/accuracy` diminuindo com dropout, o ponto em que dropout passa a causar subajuste
e a diferença de convergência entre MSE e entropia cruzada.

In [ ]:
!python src/grid_search.py grids/mlp_b3_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `mlp_b3_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("mlp_b3_regularizacao", "triagem")

In [ ]:
rep.heatmap("mlp_b3_regularizacao", row="dropout", col="loss_fn")

In [ ]:
rep.plot_grid_bars("mlp_b3_regularizacao")

#### Confirmação das finalistas e campeão — `mlp_b3_regularizacao`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("mlp_b3_regularizacao", "final")

In [ ]:
rep.show_champion("mlp_b3_regularizacao")
rep.plot_finalists("mlp_b3_regularizacao")

In [ ]:
rep.heatmap("mlp_b3_regularizacao", row="dropout", col="loss_fn", value="gap/accuracy_mean")

### 📝 Análise — Bloco 3

- **Dropout reduziu o gap? A partir de qual taxa houve subajuste?** _…_
- **MSE vs entropia cruzada: diferença de acurácia e de velocidade de convergência:** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Resultado final — teste revelado

Até aqui todas as escolhas foram feitas pela validação cruzada. Agora o conjunto de teste é usado **uma única vez**
para os campeões de cada bloco e para as referências do Bloco 0. A tabela mostra média ± desvio do teste entre os
5 modelos (um por fold) de cada configuração.

In [ ]:
final = rep.final_report(['mlp_b0_referencia', 'mlp_b1_topologia', 'mlp_b2_otimizacao', 'mlp_b2_checagem', 'mlp_b3_regularizacao'])
final

Desempenho por classe no teste: referências do Bloco 0 × campeão final. Classes com baixo recall indicam confusões sistemáticas (veja a matriz de confusão).

In [ ]:
campeao_final = rep.show_champion("mlp_b3_regularizacao")["exp_name"]
comparar = list(final.loc[final["papel"] == "referência", "exp_name"]) + [campeao_final]
rep.plot_per_class(comparar, metric="recall")
rep.plot_per_class(comparar, metric="precision")
pd.read_csv(os.path.join(os.environ["EXP_OUTPUT_DIR"], campeao_final, "matriz_confusao_teste.csv"), index_col=0)

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## 📝 Conclusões

- **Estrutura base que melhor absorveu o CIFAR-10 (Bloco 1) e por quê:** _…_
- **Ganho de otimização (Bloco 2) e sensibilidade à taxa de aprendizagem:** _…_
- **A checagem confirmou a escolha em blocos?** _…_
- **Efeito da regularização/erro (Bloco 3):** _…_
- **Ganho total sobre a referência (teste):** _…_
- **Classes mais difíceis e hipótese:** _…_